# 线性代数

在介绍完如何存储和操作数据后，接下来将简要地回顾一下部分基本线性代数内容。这些内容有助于读者了解和实现本书中介绍的大多数模型。本节将介绍线性代数中的基本数学对象、算术和运算，并用数学符号和相应的代码实现来表示它们。

---
## 环境配置

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os, sys
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import pypto
import torch
import torch_npu
import numpy as np

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
mode = pypto.RunMode.NPU

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; line-height: 1.65; color: #4b5563; font-size: 14px; background-color: #ffffff;">
    <ul style="margin:0; padding-left:20px;">
     之前只设了环境变量，.npu() 隐式把张量放到 npu:0——恰好和 TILE_FWK_DEVICE_ID=0 一致。
现在多读一步环境变量并同步 PyTorch 设备，是为了当设备号不是 0 时，PyTorch 侧的 device 仍与 PyPTO 侧保持一致。更加灵活。
    </ul>
  </div>
</details>

---
## 标量


如果你曾经在餐厅支付餐费，那么应该已经知道一些基本的线性代数，比如在数字间相加或相乘。例如，北京的温度为$52^{\circ}F$（华氏度，除摄氏度外的另一种温度计量单位）。严格来说，仅包含一个数值被称为*标量*（scalar）。如果要将此华氏度值转换为更常用的摄氏度，则可以计算表达式$c=\frac{5}{9}(f-32)$，并将$f$赋为$52$。在此等式中，每一项（$5$、$9$和$32$）都是标量值。符号$c$和$f$称为*变量*（variable），它们表示未知的标量值。

本书采用了数学表示法，其中标量变量由普通小写字母表示（例如，$x$、$y$和$z$）。本书用$\mathbb{R}$表示所有（连续）*实数*标量的空间，之后将严格定义*空间*（space）是什么，但现在只要记住表达式$x\in\mathbb{R}$是表示$x$是一个实值标量的正式形式。符号$\in$称为“属于”，它表示“是集合中的成员”。例如$x, y \in \{0,1\}$可以用来表明$x$和$y$是值只能为$0$或$1$的数字。

**在传统深度学习框架中，标量通常由仅包含单个元素的张量表示。** 例如在 PyTorch 中，标量可表示为0维 Tensor，且支持标量间的直接四则运算。然而，PyPTO 作为纯粹的张量计算框架，并未引入0维 Tensor 的概念。其运算体系仅聚焦于标量与张量、张量与张量之间的交互，不提供标量与标量间的独立运算。

以下仅演示 PyPTO 中标量的创建方式。实例化两个标量的代码如下：e1 通过 pypto.Element 构造函数直接创建；e2 则先通过 Python 原生运算得出结果，再使用 pypto.Element 进行包装。

In [4]:
e1 = pypto.Element(pypto.DT_FP32, 3)
e2 = pypto.Element(pypto.DT_FP32, 1.0 + 2.0) 
print(f"e1 = {e1.value}, e2 = {e2.value}")

e1 = 3.0, e2 = 3.0


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
x = torch.tensor(3.0)
y = torch.tensor(2.0)

x + y, x * y, x / y, x**y</pre>
    <span style="color: #1f2937; font-family: Consolas, monospace; font-size: 13px;">tensor(5.), tensor(6.), tensor(1.5000), tensor(9.)</span>
  </div>
</details>

---
## 向量

**向量可以被视为标量值组成的列表**。这些标量值被称为向量的*元素*（element）或*分量*（component）。当向量表示数据集中的样本时，它们的值具有一定的现实意义。例如，如果我们正在训练一个模型来预测贷款违约风险，可能会将每个申请人与一个向量相关联，其分量与其收入、工作年限、过往违约次数和其他因素相对应。如果我们正在研究医院患者可能面临的心脏病发作风险，可能会用一个向量来表示每个患者，其分量为最近的生命体征、胆固醇水平、每天运动时间等。在数学表示法中，向量通常记为粗体、小写的符号（例如，$\mathbf{x}$、$\mathbf{y}$和$\mathbf{z}$）。

人们通过一维张量表示向量。一般来说，张量可以具有任意长度，取决于机器的内存限制。


In [12]:
@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def arange_kernel(
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8)
    out.move(pypto.arange(0.0, 4.0, 1.0))

out = torch.zeros((4,), dtype=torch.float32, device=device)
arange_kernel(out)
out

tensor([0., 1., 2., 3.], device='npu:0')

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; line-height: 1.65; color: #4b5563; font-size: 14px; background-color: #ffffff;">
    <!-- out 赋值说明 -->
    <p style="font-weight: 600; margin: 0 0 8px 0; color: #374151; font-size: 14px;">out 赋值说明</p>
    <p style="margin: 0 0 8px 0;">把计算结果写入 <code>out</code>，有两种完全等价的写法：</p>
    <pre style="background:#fafbfb; border:1px solid #d0d7de; font-size:14px; padding:12px; border-radius:4px; font-family:monospace; color:#000;">
out[:] = result   # 空切片赋值
out.move(result)  # 显式移动</pre>
    <p style="margin: 8px 0 0 0;">其编译产物一模一样，<code>out[:] = ...</code> 底层就是调 <code>move</code>，都是零拷贝的存储重绑定，不会逐元素复制。之前章节统一用切片写法，因为 <code>[:]</code> 对 Python 用户更直觉；之后教程中两种写法都会出现，你可以按可读性自由选择。</p>
    <hr style="border: none; border-top: 1px solid #e3e3ee; margin: 16px 0;">
    <!-- arange 注意事项 -->
    <p style="font-weight: 600; margin: 0 0 8px 0; color: #374151; font-size: 14px;">arange 注意事项</p>
    <ul style="margin:0; padding-left:20px;">
      <li>默认情况下，<code>pypto.arange</code> 的 <code>start</code> 和 <code>step</code> 会被解析为 <strong>INT32</strong>。这意味着，虽然书写 <code>arange(20.0)</code> 也能得到浮点型 Tensor，但此时 <code>start</code> 和 <code>step</code> 依然是 INT32，与 FP32 类型的 <code>end</code> 产生了类型不一致，这并不是一种优雅的做法。若期望得到浮点型 Tensor，最规范的方式是采用<strong>三参数全浮点</strong>的写法。</li>
    </ul>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">x = torch.arange(4)</pre>
    <span style="color: #1f2937; font-family: Consolas, monospace; font-size: 13px;">tensor([0, 1, 2, 3])</span>
  </div>
</details>

我们可以使用下标来引用向量的任一元素，例如可以通过$x_i$来引用第$i$个元素。注意，元素$x_i$是一个标量，所以我们在引用它时不会加粗。大量文献认为列向量是向量的默认方向，在本书中也是如此。在数学中，向量$\mathbf{x}$可以写为：
<a id="eq-2-3-1"></a>

$$\mathbf{x} =\begin{bmatrix}x_{1}  \\x_{2}  \\ \vdots  \\x_{n}  \end{bmatrix}\tag{2.3.1},$$


其中$x_1,\ldots,x_n$是向量的元素。在 PyTorch 中，我们常通过诸如 x[3] 的索引来直接访问张量中的具体元素。PyPTO 虽然也支持相同的索引语法，但语义截然不同：它返回的并非一个具体的数值，而是一个 SymbolicScalar（符号标量）。

这个 SymbolicScalar 本质上是一个处于计算图中的“占位符”，代表着运行时才会确定的那个位置的数据。你无法在构建阶段去访问或打印它的实际值，但它完全可以作为节点参与后续的计算关系构建。

这正契合了 PyPTO 的设计哲学——旨在构建静态的 Tensor 关系图，而非动态求值。在“定义计算关系”的阶段，试图获取一个必须等到“运行时”才能确定的具体数值是不合理的。因此，正确的做法是让这些“占位符”在算子间流转操作，而非试图提取具体的元素值。

In [ ]:
# x[3] 返回SymbolicScalar，一个“占位符”，而非值，不可访问，但可以用于后续计算

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; line-height: 1.65; color: #4b5563; font-size: 14px; background-color: #ffffff;">
    
| 属性 | 类型 | 可写 | 描述 |
| :--- | :--- | :--- | :--- |
| `tensor.dtype` | `DataType` | 否 | 元素数据类型 |
| `tensor.shape` | `List[SymInt]` | 否 | 各轴维度（整数或 SymbolicScalar） |
| `tensor.dim` | `int` | 否 | 维度数量（秩） |
| `tensor.id` | `int` | 否 | 计算图中的唯一标识符 |
| `tensor.format` | `TileOpFormat` | 否 | 内存布局（TILEOP_ND 或 TILEOP_NZ） |
| `tensor.name` | `str` | 是 | 调试名称；可在创建后赋值 |
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">x[3]</pre>
    <span style="color: #1f2937; font-family: Consolas, monospace; font-size: 13px;">3</span>
  </div>
</details>

### 长度、维度和形状

向量只是一个数字数组，就像每个数组都有一个长度一样，每个向量也是如此。在数学表示法中，如果我们想说一个向量$\mathbf{x}$由$n$个实值标量组成，可以将其表示为$\mathbf{x}\in\mathbb{R}^n$。向量的长度通常称为向量的*维度*（dimension）。

与 Python 原生列表及 PyTorch 类似，我们通常使用内置的 `len()` 函数来获取张量长度。然而，PyPTO 的 Tensor 并未提供 `len()` 支持。原因与禁止直接访问元素值一致：对于动态 Tensor，其长度在构建期是未知的，若强行获取理应返回类似 `SymbolicScalar` 的占位符；但这与 Python 中 `len()` 必须返回非负整数的强制语义产生了根本冲突。因此，可能是处于这种原因，PyPTO 刻意规避了 `len()`。作为替代，你可以通过 `shape` 属性获取符号化的维度信息，从而在不触发实际求值的前提下完成计算关系的构建（详见下文 `shape` 的相关操作）。

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">len(x)</pre>
    <span style="color: #1f2937; font-family: Consolas, monospace; font-size: 13px;">4</span>
  </div>
</details>

当用张量表示一个向量（只有一个轴）时，我们也可以通过`.shape`属性访问向量的长度。形状（shape）是一个元素组，列出了张量沿每个轴的长度（维数）。对于**只有一个轴的张量，形状只有一个元素。**

In [8]:
@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def arange_kernel(
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8)
    out.move(pypto.arange(0.0, 4.0, 1.0))
    print(f"[JIT] out.shape = {out.shape}") # 在 JIT 装饰器中打印 out.shape

out = torch.zeros((4,), dtype=torch.float32, device=device)
arange_kernel(out)

[JIT] out.shape = [4]


<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; line-height: 1.65; color: #4b5563; font-size: 14px; background-color: #ffffff;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;">当张量包含动态轴时——无论是由构造函数传入 SymbolicScalar 标记，还是通过 from_torch(dynamic_axis=...) 指定——其 tensor.shape 对应维度的条目将不再返回普通整数，而是返回 SymbolicScalar 符号对象。</li>
    </ul>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">x.shape</pre>
    <span style="color: #1f2937; font-family: Consolas, monospace; font-size: 13px;">torch.Size([4])</span>
  </div>
</details>

请注意，*维度*（dimension）这个词在不同上下文时往往会有不同的含义，这经常会使人感到困惑。为了清楚起见，我们在此明确一下：*向量*或*轴*的维度被用来表示*向量*或*轴*的长度，即向量或轴的元素数量。然而，张量的维度用来表示张量具有的轴数。在这个意义上，张量的某个轴的维数就是这个轴的长度。

---
## 矩阵

正如向量将标量从零阶推广到一阶，矩阵将向量从一阶推广到二阶。矩阵，我们通常用粗体、大写字母来表示（例如，$\mathbf{X}$、$\mathbf{Y}$和$\mathbf{Z}$），在代码中表示为具有两个轴的张量。

数学表示法使用$\mathbf{A} \in \mathbb{R}^{m \times n}$来表示矩阵$\mathbf{A}$，其由$m$行和$n$列的实值标量组成。我们可以将任意矩阵$\mathbf{A} \in \mathbb{R}^{m \times n}$视为一个表格，其中每个元素$a_{ij}$属于第$i$行第$j$列：
<a id="eq-2-3-2"></a>

$$\mathbf{A}=\begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1n} \\ a_{21} & a_{22} & \cdots & a_{2n} \\ \vdots & \vdots & \ddots & \vdots \\ a_{m1} & a_{m2} & \cdots & a_{mn} \\  \end{bmatrix}\tag{2.3.2}.$$


对于任意$\mathbf{A} \in \mathbb{R}^{m \times n}$，$\mathbf{A}$的形状是（$m$,$n$）或$m \times n$。当矩阵具有相同数量的行和列时，其形状将变为正方形；因此，它被称为*方阵*（square matrix）。

当调用函数来实例化张量时，我们可以**通过指定两个分量$m$和$n$来创建一个形状为$m \times n$的矩阵**。


In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def reshape_arange_kernel(
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(5, 8)
    A = pypto.arange(20.0).reshape([5, 4])
    out.move(A)

out = torch.zeros((5, 4), dtype=torch.float32, device=device)
reshape_arange_kernel(out)
out

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.],
        [16., 17., 18., 19.]], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">A = torch.arange(20).reshape(5, 4)</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19]])</pre>
  </div>
</details>

在PyTorch中，我们可以通过行索引（$i$）和列索引（$j$）来访问矩阵中的标量元素$a_{ij}$，例如$[\mathbf{A}]_{ij}$。如果没有给出矩阵$\mathbf{A}$的标量元素，如[2.3.2](#eq-2-3-2)那样，我们可以简单地使用矩阵$\mathbf{A}$的小写字母索引下标$a_{ij}$来引用$[\mathbf{A}]_{ij}$。为了表示起来简单，只有在必要时才会将逗号插入到单独的索引中，例如$a_{2,3j}$和$[\mathbf{A}]_{2i-1,3}$。

当我们交换矩阵的行和列时，结果称为矩阵的*转置*（transpose）。通常用$\mathbf{A}^\top$来表示矩阵的转置，如果$\mathbf{B}=\mathbf{A}^\top$，则对于任意$i$和$j$，都有$b_{ij}=a_{ji}$。因此，在[2.3.2](#eq-2-3-2)中的转置是一个形状为$n \times m$的矩阵：
<a id="eq-2-3-3"></a>

$$
\mathbf{A}^\top =
\begin{bmatrix}
    a_{11} & a_{21} & \dots  & a_{m1} \\
    a_{12} & a_{22} & \dots  & a_{m2} \\
    \vdots & \vdots & \ddots  & \vdots \\
    a_{1n} & a_{2n} & \dots  & a_{mn} 
\end{bmatrix}\tag{2.3.3}.
$$

现在在代码中访问**矩阵的转置**。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def transpose_kernel(
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(5, 8)
    A = pypto.arange(20.0).reshape([5, 4])
    out.move(A.transpose(0, 1))

out = torch.zeros((4, 5), dtype=torch.float32, device=device)
transpose_kernel(out)
out

tensor([[ 0.,  4.,  8., 12., 16.],
        [ 1.,  5.,  9., 13., 17.],
        [ 2.,  6., 10., 14., 18.],
        [ 3.,  7., 11., 15., 19.]], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
A = torch.arange(20).reshape(5, 4)
A.T
    </pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor([[ 0,  4,  8, 12, 16],
        [ 1,  5,  9, 13, 17],
        [ 2,  6, 10, 14, 18],
        [ 3,  7, 11, 15, 19]])</pre>
  </div>
</details>

作为方阵的一种特殊类型，***对称矩阵*（symmetric matrix）$\mathbf{B}$等于其转置：$\mathbf{B} = \mathbf{B}^\top$**。这里定义一个对称矩阵$\mathbf{B} = \mathbf{A} + \mathbf{A}^\top$，然后我们将`B`与它的转置进行比较。

In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def eq_kernel(
    out: pypto.Tensor([], pypto.DT_BOOL),
):
    pypto.set_vec_tile_shapes(3, 8)
    A = pypto.arange(0.0, 9.0, 1.0).reshape([3, 3])
    B = A + A.transpose(0, 1)
    out.move(pypto.eq(B, B.transpose(0, 1)))

out = torch.zeros((3, 3), dtype=torch.bool, device=device)
eq_kernel(out)
out

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
B == B.T
    </pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor([[True, True, True],
        [True, True, True],
        [True, True, True]])</pre>
  </div>
</details>

矩阵是有用的数据结构：它们允许我们组织具有不同模式的数据。例如，我们矩阵中的行可能对应于不同的房屋（数据样本），而列可能对应于不同的属性。曾经使用过电子表格软件或已阅读过[2.2节](./02.02_data_preprocessing.ipynb)的人，应该对此很熟悉。因此，尽管单个向量的默认方向是列向量，但在表示表格数据集的矩阵中，将每个数据样本作为矩阵中的行向量更为常见。后面的章节将讲到这点，这种约定将支持常见的深度学习实践。例如，沿着张量的最外轴，我们可以访问或遍历小批量的数据样本。

---
## 张量

**就像向量是标量的推广，矩阵是向量的推广一样，我们可以构建具有更多轴的数据结构**。张量（本小节中的“张量”指代数对象）是描述具有任意数量轴的$n$维数组的通用方法。例如，向量是一阶张量，矩阵是二阶张量。张量用特殊字体的大写字母表示（例如，$\mathsf{X}$、$\mathsf{Y}$和$\mathsf{Z}$），它们的索引机制（例如$x_{ijk}$和$[\mathsf{X}]_{1,2i-1,3}$）与矩阵类似。

当我们开始处理图像时，张量将变得更加重要，图像以$n$维数组形式出现，其中3个轴对应于高度、宽度，以及一个*通道*（channel）轴，用于表示颜色通道（红色、绿色和蓝色）。现在先将高阶张量暂放一边，而是专注学习其基础知识。


In [10]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def transpose_kernel(
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(2, 3, 8)
    A = pypto.arange(24.0).reshape([2, 3, 4])
    out.move(A)

out = torch.zeros((2, 3, 4), dtype=torch.float32, device=device)
transpose_kernel(out)
out

tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]],

        [[12., 13., 14., 15.],
         [16., 17., 18., 19.],
         [20., 21., 22., 23.]]], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
X = torch.arange(24).reshape(2, 3, 4)
    </pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],
        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])</pre>
  </div>
</details>

---
## 张量算法的基本性质

标量、向量、矩阵和任意数量轴的张量（本小节中的“张量”指代数对象）有一些实用的属性。例如，从按元素操作的定义中可以注意到，任何按元素的一元运算都不会改变其操作数的形状。同样，**给定具有相同形状的任意两个张量，任何按元素二元运算的结果都将是相同形状的张量**。例如，将两个相同形状的矩阵相加，会在这两个矩阵上执行元素加法。

In [11]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def add_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    y: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(5, 8)
    out.move(pypto.add(x, y))  # 矩阵和

A = torch.arange(20, dtype=torch.float32, device=device).reshape(5, 4)
B = A.clone()
out = torch.zeros((5, 4), dtype=torch.float32, device=device)
add_kernel(A, B, out)
out

tensor([[ 0.,  2.,  4.,  6.],
        [ 8., 10., 12., 14.],
        [16., 18., 20., 22.],
        [24., 26., 28., 30.],
        [32., 34., 36., 38.]], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
A = torch.arange(20, dtype=torch.float32).reshape(5, 4)
B = A.clone()  # 通过分配新内存，将A的一个副本分配给B
C = A + B      # 矩阵和</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor([[ 0.,  2.,  4.,  6.],
        [ 8., 10., 12., 14.],
        [16., 18., 20., 22.],
        [24., 26., 28., 30.],
        [32., 34., 36., 38.]])</pre>
  </div>
</details>

具体而言，**两个矩阵的按元素乘法称为*Hadamard积*（Hadamard product，被翻译为哈达玛积）（数学符号$\odot$）**。对于矩阵$\mathbf{B} \in \mathbb{R}^{m \times n}$，其中第$i$行和第$j$列的元素是$b_{ij}$。矩阵$\mathbf{A}$（在[2.3.2](#eq-2-3-2)中定义）和$\mathbf{B}$的Hadamard积为：
<a id="eq-2-3-4"></a>

$$
\mathbf{A} \odot \mathbf{B} =
\begin{bmatrix}
    a_{11}  b_{11} & a_{12}  b_{12} & \dots  & a_{1n}  b_{1n} \\
    a_{21}  b_{21} & a_{22}  b_{22} & \dots  & a_{2n}  b_{2n} \\
    \vdots & \vdots & \ddots & \vdots \\
    a_{m1}  b_{m1} & a_{m2}  b_{m2} & \dots  & a_{mn}  b_{mn} 
\end{bmatrix}\tag{2.3.4}.
$$


In [12]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def mul_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    y: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(5, 8)
    out.move(pypto.mul(x, y))

A = torch.arange(20, dtype=torch.float32, device=device).reshape(5, 4)
B = A.clone()
out = torch.zeros((5, 4), dtype=torch.float32, device=device)
mul_kernel(A, B, out)
out

tensor([[  0.,   1.,   4.,   9.],
        [ 16.,  25.,  36.,  49.],
        [ 64.,  81., 100., 121.],
        [144., 169., 196., 225.],
        [256., 289., 324., 361.]], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
A = torch.arange(20, dtype=torch.float32).reshape(5, 4)
B = A.clone()
C = A * B</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor([[  0.,   1.,   4.,   9.],
        [ 16.,  25.,  36.,  49.],
        [ 64.,  81., 100., 121.],
        [144., 169., 196., 225.],
        [256., 289., 324., 361.]])</pre>
  </div>
</details>

张量与标量相加或相乘不会改变张量的形状，标量会与张量中的每个元素逐一进行运算。需要特别留意的是 PyPTO 的运算符顺序约束：仅有张量与标量的加法同时支持左加与右加（即 `Tensor + 标量` 与 `标量 + Tensor` 均合法）；而对于其他运算（如乘法），第一操作数必须为张量，即必须严格遵循 `Tensor 在前、标量在后` 的书写顺序，不可颠倒。

In [30]:
## 张量-标量加法
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def scalar_add_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(2, 3, 8)
    out.move(pypto.add(x, 2))

x = torch.arange(24, dtype=torch.float32, device=device).reshape(2, 3, 4)
out = torch.zeros((2, 3, 4), dtype=torch.float32, device=device)
scalar_add_kernel(x, out)
out

tensor([[[ 2.,  3.,  4.,  5.],
         [ 6.,  7.,  8.,  9.],
         [10., 11., 12., 13.]],

        [[14., 15., 16., 17.],
         [18., 19., 20., 21.],
         [22., 23., 24., 25.]]], device='npu:0')

In [29]:
## 张量-标量乘法
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def scalar_mul_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(2, 3, 8)
    out.move(pypto.mul(x, 2))

x = torch.arange(24, dtype=torch.float32, device=device).reshape(2, 3, 4)
out = torch.zeros((2, 3, 4), dtype=torch.float32, device=device)
scalar_mul_kernel(x, out)
out

tensor([[[ 0.,  2.,  4.,  6.],
         [ 8., 10., 12., 14.],
         [16., 18., 20., 22.]],

        [[24., 26., 28., 30.],
         [32., 34., 36., 38.],
         [40., 42., 44., 46.]]], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
a = 2
X = torch.arange(24).reshape(2, 3, 4)
a + X, a * X</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
(tensor([[[ 2,  3,  4,  5],
          [ 6,  7,  8,  9],
          [10, 11, 12, 13]],
         [[14, 15, 16, 17],
          [18, 19, 20, 21],
          [22, 23, 24, 25]]]),
 tensor([[[ 0,  2,  4,  6],
          [ 8, 10, 12, 14],
          [16, 18, 20, 22]],
         [[24, 26, 28, 30],
          [32, 34, 36, 38],
          [40, 42, 44, 46]]]))</pre>
  </div>
</details>

---
## 降维

我们可以对任意张量进行的一个有用的操作是**计算其元素的和**。数学表示法使用$\sum$符号表示求和。为了表示长度为$d$的向量中元素的总和，可以记为$\sum_{i=1}^dx_i$。在代码中可以调用计算求和的函数。

与 PyTorch 归约后可能产生标量（0-D Tensor）不同，PyPTO 中不存在 0-D Tensor。因此，当对最后一个维度执行归约操作（如求和）时，即使设置了 `keepdim=False`，框架也会自动保留该维度，其实际效果等同于 `keepdim=True`，始终输出形状为 [1] 的张量。在函数参数声明中，可以使用 `pypto.Tensor([], ...)` 来接收外部传入的标量，PyPTO 会在内部自动推断并桥接其与 1 维张量的映射关系。基于此设计，PyPTO 也不支持 PyTorch 中 `A.sum()` 这种不指定维度的全量求和写法，若需对多维张量求全量之和，必须显式指定维度逐维进行归约。

In [15]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def sum_vec_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8, 8)
    result = pypto.sum(x, dim=0, keepdim=False)
    print(f"[JIT] result.shape = {result.shape}") # 尽管使用了 keepdim = False 条件，但result仍然是1维 Tensor，这从另一个角度体现PyPTO中不存在0-D张量
    out.move(result)

x = torch.arange(4.0, dtype=torch.float32, device=device)
out = torch.empty((), dtype=torch.float32, device=device)
sum_vec_kernel(x, out)
out

[JIT] result.shape = [1]


tensor(6., device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
x = torch.arange(4, dtype=torch.float32)
x.sum()</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor(6.)</pre>
  </div>
</details>

我们可以**表示任意形状张量的元素和**。
例如，矩阵$\mathbf{A}$中元素的和可以记为$\sum_{i=1}^{m} \sum_{j=1}^{n} a_{ij}$。

In [16]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def sum_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8, 8)
    out.move(pypto.sum(x, dim=1, keepdim=False).sum(dim=0, keepdim=False))

x = torch.arange(20.0, dtype=torch.float32, device=device).reshape(5, 4)
out = torch.empty((1,), dtype=torch.float32, device=device)
sum_kernel(x, out)
out

tensor([190.], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
A = torch.arange(20.0).reshape(5, 4)
A.sum()
    </pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor(190.)</pre>
  </div>
</details>

默认情况下，调用求和函数会沿所有的轴降低张量的维度，使它变为一个标量。我们还可以**指定张量沿哪一个轴来通过求和降低维度**。以矩阵为例，为了通过求和所有行的元素来降维（轴0），可以在调用函数时指定`axis=0`。由于输入矩阵沿0轴降维以生成输出向量，因此输入轴0的维数在输出形状中消失。

还可以指定`axis=1`将通过汇总所有列的元素降维（轴1）。因此，输入轴1的维数在输出形状中消失。

同样的，沿着行和列对矩阵求和，等价于对矩阵的所有元素进行求和。

In [17]:
# 沿轴0（垂直方向）求和，压缩掉行维度，得到各列的总和
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def sum_axis0_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8, 8)
    out.move(pypto.sum(x, dim=0, keepdim=False))

x = torch.arange(20.0, dtype=torch.float32, device=device).reshape(5, 4)
out = torch.empty((4,), dtype=torch.float32, device=device)
sum_axis0_kernel(x, out)
out

tensor([40., 45., 50., 55.], device='npu:0')

In [ ]:
# 沿轴1（水平方向）求和，压缩掉列维度，得到各行的总和
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def sum_axis1_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8, 8)
    out.move(pypto.sum(x, dim=1, keepdim=False))

x = torch.arange(20.0, dtype=torch.float32, device=device).reshape(5, 4)
out = torch.empty((5,), dtype=torch.float32, device=device)
sum_axis1_kernel(x, out)
out

tensor([ 6., 22., 38., 54., 70.], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
A = torch.arange(20.0, dtype=torch.float32, device=device).reshape(5, 4)
A_sum_axis0 = A.sum(axis=0)  # 沿轴0（垂直方向）求和，压缩掉行维度，得到各列的总和
A_sum_axis1 = A.sum(axis=1)  # 沿轴1（水平方向）求和，压缩掉列维度，得到各行的总和
A.sum(axis=[0, 1])           # 沿轴0和轴1同时求和，即求矩阵所有元素的总和，返回标量，结果和A.sum()相同</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor([40., 45., 50., 55.])
tensor([ 6., 22., 38., 54., 70.])
tensor(190.)</pre>
  </div>
</details>

**一个与求和相关的量是*平均值*（mean或average）**。
我们通过将总和除以元素总数来计算平均值。
在代码中，我们可以调用函数来计算任意形状张量的平均值。

In [19]:
# 静态形状张量的均值
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def mean_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(4, 8)
    total = pypto.sum(a, dim=-1, keepdim=False)
    pypto.set_vec_tile_shapes(8) # tile shape 的维数必须和当前张量的秩一致
    total = pypto.sum(total, dim=0, keepdim=False)
    out.move(pypto.div(total, 20.0))

a = torch.arange(20.0, dtype=torch.float32, device=device).reshape(5, 4)
out = torch.empty((), dtype=torch.float32, device=device)
mean_kernel(a, out)
out

tensor(9.5000, device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
A = torch.arange(20.0).reshape(5, 4)
A.mean(), A.sum() / A.numel()</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor(9.500), tensor(9.500)</pre>
  </div>
</details>

同样，计算平均值的函数也可以沿指定轴降低张量的维度。

In [20]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def mean0_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8, 8)
    temp = pypto.sum(x, dim=0, keepdim=False)
    out.move(pypto.div(temp, x.shape[0]))

x = torch.arange(20.0, dtype=torch.float32, device=device).reshape(5, 4)
out = torch.empty((4,), dtype=torch.float32, device=device)
mean0_kernel(x, out)
out

tensor([ 8.,  9., 10., 11.], device='npu:0')

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; line-height: 1.65; color: #4b5563; font-size: 14px; background-color: #ffffff;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;">至于动态类型，pypto.div不接受SymbolicScalar作为参数...</li>
    </ul>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">A = torch.arange(20.0).reshape(5, 4)
A.mean(axis=0), A.sum(axis=0) / A.shape[0]</pre>
    <span style="color: #1f2937; font-family: Consolas, monospace; font-size: 13px;">(tensor([8., 9., 10., 11.]), tensor([8., 9., 10., 11.]))</span>
  </div>
</details>

### 非降维求和

但是，有时在调用函数来**计算总和或均值时保持轴数不变**会很有用。

例如，由于`sum_A`在对每行进行求和后仍保持两个轴，我们可以(**通过广播将`A`除以`sum_A`**)。

In [21]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def row_normalize_kernel(
    A: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8, 8)
    sum_A = pypto.sum(A, 1, keepdim=True)   
    out.move(pypto.div(A, sum_A))         
 
A = torch.arange(20.0, dtype=torch.float32, device=device).reshape(5, 4)
out = torch.empty((5, 4), dtype=torch.float32, device=device)
row_normalize_kernel(A, out)
out

tensor([[0.0000, 0.1667, 0.3333, 0.5000],
        [0.1818, 0.2273, 0.2727, 0.3182],
        [0.2105, 0.2368, 0.2632, 0.2895],
        [0.2222, 0.2407, 0.2593, 0.2778],
        [0.2286, 0.2429, 0.2571, 0.2714]], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
A = torch.arange(20.0).reshape(5, 4)
sum_A = A.sum(axis=1, keepdims=True)
A / sum_A     # 例如，由于 sum_A 在每行求和后仍保持两个轴，可以通过广播将 A 除以 sum_A</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor([[0.0000, 0.1667, 0.3333, 0.5000],
        [0.1818, 0.2273, 0.2727, 0.3182],
        [0.2105, 0.2368, 0.2632, 0.2895],
        [0.2222, 0.2407, 0.2593, 0.2778],
        [0.2286, 0.2429, 0.2571, 0.2714]])</pre>
  </div>
</details>

如果我们想沿**某个轴计算`A`元素的累积总和**，
比如`axis=0`（按行计算），可以调用`cumsum`函数。
此函数不会沿任何轴降低输入张量的维度。

In [22]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def cumsum_kernel(
    A: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(5, 8)
    out.move(pypto.cumsum(A, 0))

A = torch.arange(20.0, dtype=torch.float32, device=device).reshape(5, 4)
out = torch.empty(5, 4, dtype=torch.float32, device=device)
cumsum_kernel(A, out)
out

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  6.,  8., 10.],
        [12., 15., 18., 21.],
        [24., 28., 32., 36.],
        [40., 45., 50., 55.]], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
A = torch.arange(20.0, dtype=torch.float32, device=device).reshape(5, 4)
A.cumsum(axis=0)</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  6.,  8., 10.],
        [12., 15., 18., 21.],
        [24., 28., 32., 36.],
        [40., 45., 50., 55.]])</pre>
  </div>
</details>

---
## 点积（Dot Product）

我们已经学习了按元素操作、求和及平均值。另一个最基本的操作之一是点积。给定两个向量$\mathbf{x},\mathbf{y}\in\mathbb{R}^d$，它们的*点积*（dot product）$\mathbf{x}^\top\mathbf{y}$（或$\langle\mathbf{x},\mathbf{y}\rangle$）是相同位置的按元素乘积的和：$\mathbf{x}^\top \mathbf{y} = \sum_{i=1}^{d} x_i y_i$。

In [23]:
# PyPTO点乘的唯一方案：逐元素乘 + 求和
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def dot_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    y: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8, 8)
    out.move(pypto.sum(x * y, 0, keepdim=True))

x = torch.arange(4.0, dtype=torch.float32, device=device)
y = torch.ones(4, dtype = torch.float32, device=device)
out = torch.empty(1, dtype=torch.float32, device=device)
dot_kernel(x, y, out)
x, y, out

(tensor([0., 1., 2., 3.], device='npu:0'),
 tensor([1., 1., 1., 1.], device='npu:0'),
 tensor([6.], device='npu:0'))

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
x = torch.arange(4.0, dtype=torch.float32)
y = torch.ones(4, dtype = torch.float32)
x, y, torch.dot(x, y), torch.sum(x * y) # 注意，我们也可以通过执行按元素乘法，然后进行求和来表示两个向量的点积</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
(tensor([0., 1., 2., 3.]),
 tensor([1., 1., 1., 1.]),
 tensor(6.),
 tensor(6.))</pre>
  </div>
</details>

点积在很多场合都很有用。例如，给定一组由向量$\mathbf{x} \in \mathbb{R}^d$表示的值，和一组由$\mathbf{w} \in \mathbb{R}^d$表示的权重。$\mathbf{x}$中的值根据权重$\mathbf{w}$的加权和，可以表示为点积$\mathbf{x}^\top \mathbf{w}$。当权重为非负数且和为1（即$\left(\sum_{i=1}^{d}{w_i}=1\right)$）时，点积表示*加权平均*（weighted average）。将两个向量规范化得到单位长度后，点积表示它们夹角的余弦。本节后面的内容将正式介绍*长度*（length）的概念。

---
## 矩阵-向量积

现在我们知道如何计算点积，可以开始理解*矩阵-向量积*（matrix-vector product）。回顾分别在[2.3.2](#eq-2-3-2)和[2.3.1](#eq-2-3-1)中定义的矩阵$\mathbf{A} \in \mathbb{R}^{m \times n}$和向量$\mathbf{x} \in \mathbb{R}^n$。让我们将矩阵$\mathbf{A}$用它的行向量表示：
<a id="eq-2-3-5"></a>

$$\mathbf{A}=
\begin{bmatrix}
\mathbf{a}^\top_{1} \\
\mathbf{a}^\top_{2} \\
\vdots \\
\mathbf{a}^\top_m \\
\end{bmatrix}\tag{2.3.5},$$

其中每个$\mathbf{a}^\top_{i} \in \mathbb{R}^n$都是行向量，表示矩阵的第$i$行。**矩阵向量积$\mathbf{A}\mathbf{x}$是一个长度为$m$的列向量，其第$i$个元素是点积$\mathbf{a}^\top_i \mathbf{x}$**：
<a id="eq-2-3-6"></a>

$$
\mathbf{A}\mathbf{x}
= \begin{bmatrix}
\mathbf{a}^\top_{1} \\
\mathbf{a}^\top_{2} \\
\vdots \\
\mathbf{a}^\top_m \\
\end{bmatrix}\mathbf{x}
= \begin{bmatrix}
 \mathbf{a}^\top_{1} \mathbf{x}  \\
 \mathbf{a}^\top_{2} \mathbf{x} \\
\vdots\\
 \mathbf{a}^\top_{m} \mathbf{x}\\ 
\end{bmatrix}\tag{2.3.6}.
$$

我们可以把一个矩阵$\mathbf{A} \in \mathbb{R}^{m \times n}$乘法看作一个从$\mathbb{R}^{n}$到$\mathbb{R}^{m}$向量的转换。这些转换是非常有用的，例如可以用方阵的乘法来表示旋转。后续章节将讲到，我们也可以使用矩阵-向量积来描述在给定前一层的值时，求解神经网络每一层所需的复杂计算。

在代码中使用张量表示矩阵-向量积，我们使用`mv`函数。当我们为矩阵`A`和向量`x`调用`torch.mv(A, x)`时，会执行矩阵-向量积。注意，`A`的列维数（沿轴1的长度）必须与`x`的维数（其长度）相同。

In [24]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def mv_kernel(
    A: pypto.Tensor([], pypto.DT_FP32),
    x: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    x_col = pypto.reshape(x, [4, 1])
    result = pypto.matmul(A, x_col, pypto.DT_FP32)
    out.move(pypto.reshape(result, [5]))

A = torch.arange(20.0, dtype=torch.float32, device=device).reshape(5, 4)
x = torch.arange(4.0, dtype=torch.float32, device=device)
out = torch.empty(5, dtype=torch.float32, device=device)
mv_kernel(A, x, out)
out

tensor([ 14.,  38.,  62.,  86., 110.], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
A = torch.arange(20.0, dtype=torch.float32).reshape(5, 4)
x = torch.arange(4.0, dtype=torch.float32)
torch.mv(A, x)</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor([ 14.,  38.,  62.,  86., 110.])</pre>
  </div>
</details>

## 矩阵-矩阵乘法

在掌握点积和矩阵-向量积的知识后，那么**矩阵-矩阵乘法**（matrix-matrix multiplication）应该很简单。

假设有两个矩阵$\mathbf{A} \in \mathbb{R}^{n \times k}$和$\mathbf{B} \in \mathbb{R}^{k \times m}$：
<a id="eq-2-3-7"></a>

$$\mathbf{A}=\begin{bmatrix}
 a_{11} & a_{12} & \cdots & a_{1k} \\
 a_{21} & a_{22} & \cdots & a_{2k} \\
\vdots & \vdots & \ddots & \vdots \\
 a_{n1} & a_{n2} & \cdots & a_{nk} \\
\end{bmatrix},\quad
\mathbf{B}=\begin{bmatrix}
 b_{11} & b_{12} & \cdots & b_{1m} \\
 b_{21} & b_{22} & \cdots & b_{2m} \\
\vdots & \vdots & \ddots & \vdots \\
 b_{k1} & b_{k2} & \cdots & b_{km} \\ 
\end{bmatrix}\tag{2.3.7}.$$

用行向量$\mathbf{a}^\top_{i} \in \mathbb{R}^k$表示矩阵$\mathbf{A}$的第$i$行，并让列向量$\mathbf{b}_{j} \in \mathbb{R}^k$作为矩阵$\mathbf{B}$的第$j$列。要生成矩阵积$\mathbf{C} = \mathbf{A}\mathbf{B}$，最简单的方法是考虑$\mathbf{A}$的行向量和$\mathbf{B}$的列向量:
<a id="eq-2-3-8"></a>
$$\mathbf{A}=
\begin{bmatrix}
\mathbf{a}^\top_{1} \\
\mathbf{a}^\top_{2} \\
\vdots \\
\mathbf{a}^\top_n \\
\end{bmatrix},
\quad \mathbf{B}=\begin{bmatrix}
 \mathbf{b}_{1} & \mathbf{b}_{2} & \cdots & \mathbf{b}_{m} \\ 
\end{bmatrix}.\tag{2.3.8}
$$
当我们简单地将每个元素$c_{ij}$计算为点积$\mathbf{a}^\top_i \mathbf{b}_j$:
<a id="eq-2-3-9"></a>

$$\mathbf{C} = \mathbf{AB} = \begin{bmatrix}
\mathbf{a}^\top_{1} \\
\mathbf{a}^\top_{2} \\
\vdots \\
\mathbf{a}^\top_n \\
\end{bmatrix}
\begin{bmatrix}
 \mathbf{b}_{1} & \mathbf{b}_{2} & \cdots & \mathbf{b}_{m} \\
\end{bmatrix}
= \begin{bmatrix}
\mathbf{a}^\top_{1} \mathbf{b}_1 & \mathbf{a}^\top_{1}\mathbf{b}_2& \cdots & \mathbf{a}^\top_{1} \mathbf{b}_m \\
 \mathbf{a}^\top_{2}\mathbf{b}_1 & \mathbf{a}^\top_{2} \mathbf{b}_2 & \cdots & \mathbf{a}^\top_{2} \mathbf{b}_m \\
 \vdots & \vdots & \ddots &\vdots\\
\mathbf{a}^\top_{n} \mathbf{b}_1 & \mathbf{a}^\top_{n}\mathbf{b}_2& \cdots& \mathbf{a}^\top_{n} \mathbf{b}_m 
\end{bmatrix}\tag{2.3.9}.
$$

**我们可以将矩阵-矩阵乘法$\mathbf{AB}$看作简单地执行$m$次矩阵-向量积，并将结果拼接在一起，形成一个$n \times m$矩阵**。在下面的代码中，我们在`A`和`B`上执行矩阵乘法。这里的`A`是一个5行4列的矩阵，`B`是一个4行3列的矩阵。两者相乘后，我们得到了一个5行3列的矩阵。

In [25]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def matmul_kernel(
    x: pypto.Tensor([], pypto.DT_FP32),
    y: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    out.move(pypto.matmul(x, y, x.dtype))
 
dtype = torch.float32
A = torch.arange(20.0, dtype=torch.float32, device=device).reshape(5, 4)
B = torch.ones(4, 3, dtype=torch.float32, device=device)
out = torch.empty(5, 3, dtype=dtype, device=device)
matmul_kernel(A, B, out)
out

tensor([[ 6.,  6.,  6.],
        [22., 22., 22.],
        [38., 38., 38.],
        [54., 54., 54.],
        [70., 70., 70.]], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
A = torch.arange(20.0, dtype=torch.float32).reshape(5, 4)
B = torch.ones(4, 3)
torch.mm(A, B)</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor([[ 6.,  6.,  6.],
        [22., 22., 22.],
        [38., 38., 38.],
        [54., 54., 54.],
        [70., 70., 70.]])</pre>
  </div>
</details>

矩阵-矩阵乘法可以简单地称为**矩阵乘法**，不应与"Hadamard积"混淆。

---
## 范数


线性代数中最有用的一些运算符是*范数*（norm）。非正式地说，向量的*范数*是表示一个向量有多大。这里考虑的*大小*（size）概念不涉及维度，而是分量的大小。

在线性代数中，向量范数是将向量映射到标量的函数$f$。给定任意向量$\mathbf{x}$，向量范数要满足一些属性。第一个性质是：如果我们按常数因子$\alpha$缩放向量的所有元素，其范数也会按相同常数因子的*绝对值*缩放：
<a id="eq-2-3-10"></a>

$$f(\alpha \mathbf{x}) = |\alpha| f(\mathbf{x}). \tag{2.3.10}$$ 

第二个性质是熟悉的三角不等式:
<a id="eq-2-3-11"></a>

$$f(\mathbf{x} + \mathbf{y}) \leq f(\mathbf{x}) + f(\mathbf{y}). \tag{2.3.11}$$

第三个性质简单地说范数必须是非负的:
<a id="eq-2-3-12"></a>

$$f(\mathbf{x}) \geq 0. \tag{2.3.12}$$

这是有道理的。因为在大多数情况下，任何东西的最小的*大小*是0。最后一个性质要求范数最小为0，当且仅当向量全由0组成。
<a id="eq-2-3-13"></a>

$$\forall i, [\mathbf{x}]_i = 0 \Leftrightarrow f(\mathbf{x})=0. \tag{2.3.13}$$

范数听起来很像距离的度量。欧几里得距离和毕达哥拉斯定理中的非负性概念和三角不等式可能会给出一些启发。事实上，欧几里得距离是一个$L_2$范数：假设$n$维向量$\mathbf{x}$中的元素是$x_1,\ldots,x_n$，其 **$L_2$范数是向量元素平方和的平方根：**
<a id="eq-2-3-14"></a>

**$$\|\mathbf{x}\|_2 = \sqrt{\sum_{i=1}^n x_i^2}, \tag{2.3.14}$$**

其中，在$L_2$范数中常常省略下标$2$，也就是说$\|\mathbf{x}\|$等同于$\|\mathbf{x}\|_2$。在代码中，我们可以按如下方式计算向量的$L_2$范数。

In [26]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def l2_norm_kernel(
    u: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8, 8)
    sq_sum = pypto.sum(u * u, 0, keepdim=True)
    out.move(sq_sum.sqrt())

u = torch.tensor([3.0, -4.0], dtype=torch.float32, device=device)
out = torch.empty(1, dtype=torch.float32, device=device)
l2_norm_kernel(u, out)
out

tensor([5.], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
u = torch.tensor([3.0, -4.0])
torch.norm(u)</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor(5.)</pre>
  </div>
</details>

深度学习中更经常地使用$L_2$范数的平方，也会经常遇到 **$L_1$范数，它表示为向量元素的绝对值之和：**
<a id="eq-2-3-15"></a>

**$$\|\mathbf{x}\|_1 = \sum_{i=1}^n \left|x_i \right|.  \tag{2.3.15}$$**

与$L_2$范数相比，$L_1$范数受异常值的影响较小。为了计算$L_1$范数，我们将绝对值函数和按元素求和组合起来。

In [27]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def l1_norm_kernel(
    u: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(8)
    abs_sum = pypto.sum(pypto.abs(u), 0, keepdim=True)
    out.move(abs_sum)
 
u = torch.tensor([3.0, -4.0], dtype=torch.float32, device=device)
out = torch.empty(1, dtype=torch.float32, device=device)
l1_norm_kernel(u, out)
out

tensor([7.], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
u = torch.tensor([3.0, -4.0])
torch.abs(u).sum()</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor(7.)</pre>
  </div>
</details>

$L_2$范数和$L_1$范数都是更一般的$L_p$范数的特例：
<a id="eq-2-3-16"></a>

$$\|\mathbf{x}\|_p = \left(\sum_{i=1}^n \left|x_i \right|^p \right)^{1/p}. \tag{2.3.16}$$

类似于向量的$L_2$范数，**矩阵**$\mathbf{X} \in \mathbb{R}^{m \times n}$**的*Frobenius范数*（Frobenius norm）是矩阵元素平方和的平方根：**
<a id="eq-2-3-17"></a>

**$$\|\mathbf{X}\|_F = \sqrt{\sum_{i=1}^m \sum_{j=1}^n x_{ij}^2}. \tag{2.3.17}$$**

Frobenius范数满足向量范数的所有性质，它就像是矩阵形向量的$L_2$范数。调用以下函数将计算矩阵的Frobenius范数。

In [28]:
@pypto.frontend.jit(runtime_options={"run_mode": mode})
def frob_norm_kernel(
    A: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(4, 16)
    sq = A * A
    row_sum = pypto.sum(sq, 1, keepdim=True)   
    total = pypto.sum(row_sum, 0, keepdim=True)  
    pypto.set_vec_tile_shapes(8)                 
    total_flat = pypto.reshape(total, [1])
    out.move(total_flat.sqrt())
 
A = torch.ones((4, 9), dtype=torch.float32, device=device)
out = torch.empty(1, dtype=torch.float32, device=device)
frob_norm_kernel(A, out)
out

tensor([6.], device='npu:0')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
torch.norm(torch.ones((4, 9)))</pre>
    <pre style="color: #1f2937;font-family: Consolas, monospace; font-size: 13px;">
tensor(6.)</pre>
  </div>
</details>

### 范数和目标


在深度学习中，我们经常试图解决优化问题：*最大化*分配给观测数据的概率;*最小化*预测和真实观测之间的距离。用向量表示物品（如单词、产品或新闻文章），以便最小化相似项目之间的距离，最大化不同项目之间的距离。目标，或许是深度学习算法最重要的组成部分（除了数据），通常被表达为范数。

---
## 关于线性代数的更多信息

仅用一节，我们就教会了阅读本书所需的、用以理解现代深度学习的线性代数。线性代数还有很多，其中很多数学对于机器学习非常有用。例如，矩阵可以分解为因子，这些分解可以显示真实世界数据集中的低维结构。机器学习的整个子领域都侧重于使用矩阵分解及其向高阶张量的泛化，来发现数据集中的结构并解决预测问题。当开始动手尝试并在真实数据集上应用了有效的机器学习模型，你会更倾向于学习更多数学。因此，这一节到此结束，本书将在后面介绍更多数学知识。

如果渴望了解有关线性代数的更多信息，可以参考[线性代数运算的在线附录](https://d2l.ai/chapter_appendix-mathematics-for-deep-learning/geometry-linear-algebraic-ops.html)或其他优秀资源。


---

## 小结

* 标量、向量、矩阵和张量是线性代数中的基本数学对象。
* 向量泛化自标量，矩阵泛化自向量。
* 标量、向量、矩阵和张量分别具有零、一、二和任意数量的轴。
* 两个矩阵的按元素乘法被称为他们的Hadamard积。它与矩阵乘法不同。
* 在深度学习中，我们经常使用范数，如$L_1$范数、$L_2$范数和Frobenius范数。
* 我们可以对标量、向量、矩阵和张量执行各种操作。

---
## 练习

1. 证明一个矩阵$\mathbf{A}$的转置的转置是$\mathbf{A}$，即$(\mathbf{A}^\top)^\top = \mathbf{A}$。
1. 给出两个矩阵$\mathbf{A}$和$\mathbf{B}$，证明“它们转置的和”等于“它们和的转置”，即$\mathbf{A}^\top + \mathbf{B}^\top = (\mathbf{A} + \mathbf{B})^\top$。
1. 给定任意方阵$\mathbf{A}$，$\mathbf{A} + \mathbf{A}^\top$总是对称的吗?为什么?
1. 本节中定义了形状$(2,3,4)$的张量`X`。`len(X)`的输出结果是什么？
1. 对于任意形状的张量`X`,`len(X)`是否总是对应于`X`特定轴的长度?这个轴是什么?
1. 运行`A/A.sum(axis=1)`，看看会发生什么。请分析一下原因？
1. 考虑一个具有形状$(2,3,4)$的张量，在轴0、1、2上的求和输出是什么形状?
1. 为`linalg.norm`函数提供3个或更多轴的张量，并观察其输出。对于任意形状的张量这个函数计算得到什么

### 参考答案
参考答案位于[answers/02.03_reference_answer.ipynb](./answers/02.03_reference_answer.ipynb)